# Nível 1 — Dados e primeira análise com LLM

Este notebook implementa o tratamento dos dados, a normalização dos valores para BRL, as regras determinísticas de sinalização e a análise de um caso selecionado com LLM.



## Carga dos dados

Nesta etapa, carregamos o arquivo `dados_nivel_1.json`, a taxa de câmbio fornecida e as operações em um DataFrame pandas.

In [6]:
import pandas as pd
import json

with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

taxa_cambio = dados['taxa_cambio_usd_brl']
df = pd.DataFrame(dados['operacoes'])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio}")
print(f"Total de operações carregadas: {len(df)}")
df.head(10)

Taxa de câmbio USD/BRL: 5.4
Total de operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


## Limpeza e tratamento dos dados

Antes de qualquer agregação ou regra, vamos investigar a qualidade dos dados brutos. 
Vamos checar: registros duplicados, valores em moeda diferente de BRL, e datas ausentes.

In [3]:
import numpy as np

# Cria uma cópia para preservar o dataframe original carregado
df_clean = df.copy()

# 1. Tratamento: Remoção de duplicatas exatas pelo 'id'
df_clean = df_clean.drop_duplicates(subset=['id'], keep='first')

# 2. Tratamento: Normalização de valores para BRL
df_clean['valor_brl'] = np.where(
    df_clean['moeda'] == 'USD',
    df_clean['valor'] * taxa_cambio,
    df_clean['valor']
)

# 3. Tratamento: Conversão de datas (valores nulos viram NaT nativamente)
df_clean['data'] = pd.to_datetime(df_clean['data'])

print(f"Total de operações após limpeza: {len(df_clean)}")

# Mostrando especificamente os casos tratados para validar a limpeza
casos_tratados = ['OP-0007', 'OP-0013', 'OP-0017']
df_clean[df_clean['id'].isin(casos_tratados)][['id', 'cliente_id', 'data', 'valor', 'moeda', 'valor_brl']]

Total de operações após limpeza: 19


,id,cliente_id,data,valor,moeda,valor_brl
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,17200.0
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,64800.0
17,OP-0017,CLI-A-5,NaT,4300,BRL,4300.0


### Justificativa das Decisões de Tratamento

Conforme os problemas levantados, tomamos as seguintes decisões:

1. **Registros Duplicados (`OP-0007`):** Removemos a duplicata mantendo apenas a primeira ocorrência. Um sistema transacional não deve ter o mesmo ID de operação para transações distintas.
2. **Moeda Estrangeira (`OP-0013` em USD):** Criamos a coluna `valor_brl`. Multiplicamos o valor original pela taxa de câmbio fixa (5.4) fornecida no arquivo, normalizando a base para as regras futuras.
3. **Data Faltante (`OP-0017` com data nula):** Ao converter para datetime, o valor se tornou `NaT`. Decidimos **não excluir** a linha. Isso garante que o volume financeiro total deste cliente não seja prejudicado na etapa de agregações. A operação apenas não será contabilizada em regras que dependam estritamente do dia (como a Regra 1).

## Agregações

Com os dados tratados e os valores normalizados para BRL, calculamos o volume total transacionado por cliente e a quantidade de operações por canal.

In [8]:
volume_por_cliente = (
    df_clean.groupby("cliente_id")["valor_brl"]
    .sum()
    .sort_values(ascending=False)
)

print("Volume total transacionado por cliente:")
display(volume_por_cliente)

operacoes_por_canal = (
    df_clean.groupby("canal")
    .size()
    .sort_values(ascending=False)
)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Volume total transacionado por cliente:


cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor_brl, dtype: float64

Quantidade de operações por canal:


canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
dtype: int64

## Regras determinísticas

Implementamos as duas regras de sinalização exigidas, adicionando as flags ao DataFrame.
Todo o cálculo (soma, mediana, comparação com limite) é feito em pandas, a LLM não participa dessa etapa.

In [4]:
# ===== Regra 1 — Fracionamento =====
# Cliente com 3+ operações na mesma data, soma > R$50.000,
# e nenhuma operação isolada >= R$20.000

agrupado = df_clean.groupby(['cliente_id', 'data'])['valor_brl'].agg(
    qtd_operacoes='count',
    soma_valores='sum',
    maior_operacao='max'
).reset_index()

agrupado['flag_fracionamento'] = (
    (agrupado['qtd_operacoes'] >= 3) &
    (agrupado['soma_valores'] > 50000) &
    (agrupado['maior_operacao'] < 20000)
)

clientes_fracionamento = agrupado[agrupado['flag_fracionamento']]['cliente_id'].unique()
print("Clientes sinalizados na Regra 1 (Fracionamento):")
print(clientes_fracionamento)
print()

# Adiciona a flag ao df_clean (nível cliente+data)
df_clean = df_clean.merge(
    agrupado[['cliente_id', 'data', 'flag_fracionamento']],
    on=['cliente_id', 'data'],
    how='left'
)

# ===== Regra 2 — Valor atípico =====
# Operação > 5x a mediana do cliente, só para clientes com 4+ operações

contagem_cliente = df_clean.groupby('cliente_id')['valor_brl'].transform('count')
mediana_cliente = df_clean.groupby('cliente_id')['valor_brl'].transform('median')

df_clean['flag_valor_atipico'] = (
    (contagem_cliente >= 4) &
    (df_clean['valor_brl'] > 5 * mediana_cliente)
)

print("Operações sinalizadas na Regra 2 (Valor atípico):")
print(df_clean[df_clean['flag_valor_atipico']][['id', 'cliente_id', 'valor_brl']])

Clientes sinalizados na Regra 1 (Fracionamento):
<StringArray>
['CLI-A-1']
Length: 1, dtype: str

Operações sinalizadas na Regra 2 (Valor atípico):
         id cliente_id  valor_brl
12  OP-0013    CLI-A-4    64800.0


### Validação da Regra 1 (Fracionamento)

Comparamos dois casos parecidos para mostrar que a regra funciona corretamente:
CLI-A-1 (deveria ser sinalizado) e CLI-A-3 (não deveria, mesmo tendo padrão similar).

In [9]:
casos_validacao = agrupado[
    (agrupado['cliente_id'].isin(['CLI-A-1', 'CLI-A-3'])) &
    (agrupado['qtd_operacoes'] >= 3)
]

print(casos_validacao[
    ['cliente_id', 'data', 'qtd_operacoes', 'soma_valores',
     'maior_operacao', 'flag_fracionamento']
])

  cliente_id       data  qtd_operacoes  soma_valores  maior_operacao  \
0    CLI-A-1 2026-03-09              3       54200.0         18800.0   
3    CLI-A-3 2026-03-05              3       48500.0         17200.0   

   flag_fracionamento  
0                True  
3               False  
